In [ ]:
!pip install transformers datasets

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from transformers import AutoImageProcessor, ViTForImageClassification
from datasets import load_dataset
from transformers.models.vit.modeling_vit import ViTConfig, ViTSelfAttention

In [ ]:
dataset = load_dataset("huggingface/cats-image", trust_remote_code=True)
image = dataset["test"]["image"][0]

image_processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
pretrained_model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224")

inputs = image_processor(image, return_tensors="pt")

with torch.no_grad():
    logits = pretrained_model(**inputs).logits

# pretrained_model predicts one of the 1000 ImageNet classes
predicted_label = logits.argmax(-1).item()
print(pretrained_model.config.id2label[predicted_label])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Egyptian cat


In [ ]:
class FactorizedTensor(nn.Module):
    """Base class for factorized tensor adapters (Q, K, V only)"""
    def __init__(self, hidden_size: int, rank: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.rank = rank

        # Initialize base parameters (to be overridden by subclasses)
        self.register_parameter('dummy', nn.Parameter(torch.empty(0)))

    def forward(self) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Return delta matrices for Q, K, V"""
        raise NotImplementedError

    def reset_deltas_to_zero(self):
        """For testing: initialize factorization to produce zero deltas"""
        raise NotImplementedError

class CPFactorizedTensor(FactorizedTensor):
    """Canonical Polyadic (CP) Decomposition"""
    def __init__(self, hidden_size: int, rank: int):
        super().__init__(hidden_size, rank)

        # CP decomposition parameters (Q, K, V share core factors)
        self.A = nn.Parameter(torch.zeros(3, rank))  # One row per matrix (Q,K,V)
        self.B = nn.Parameter(torch.zeros(hidden_size, rank))
        self.C = nn.Parameter(torch.zeros(hidden_size, rank))

        # Initialize with small values
        nn.init.normal_(self.A, std=1.0 / (rank ** 0.5))
        nn.init.normal_(self.B, std=1.0 / (rank ** 0.5))
        nn.init.normal_(self.C, std=1.0 / (rank ** 0.5))

    def forward(self):
        """Reconstruct delta matrices using CP decomposition"""
        # For each matrix (Q,K,V), compute sum_r (A_r * B_r ⊗ C_r)
        delta_q = torch.einsum("r,ir,jr->ij", self.A[0], self.B, self.C)
        delta_k = torch.einsum("r,ir,jr->ij", self.A[1], self.B, self.C)
        delta_v = torch.einsum("r,ir,jr->ij", self.A[2], self.B, self.C)

        return delta_q, delta_k, delta_v

    def reset_deltas_to_zero(self):
        nn.init.zeros_(self.A)
        nn.init.zeros_(self.B)
        nn.init.zeros_(self.C)

class ModifiedViTSdpaSelfAttention(ViTSelfAttention):
    def __init__(self, config: ViTConfig, factorization: str = "cp", rank: int = 8):
        super().__init__(config)
        self.attention_probs_dropout_prob = config.attention_probs_dropout_prob

        # Freeze original parameters
        self.query.weight.requires_grad_(False)
        self.key.weight.requires_grad_(False)
        self.value.weight.requires_grad_(False)

        # Initialize factorization
        if factorization == "cp":
            self.factorization = CPFactorizedTensor(
                hidden_size=config.hidden_size,
                rank=rank
            )
        else:
            raise ValueError(f"Unsupported factorization: {factorization}")

    def forward(self, hidden_states, head_mask=None, output_attentions=False):
        if output_attentions or head_mask is not None:
            return super().forward(hidden_states, head_mask, output_attentions)

        # Get delta matrices from factorization
        delta_q, delta_k, delta_v = self.factorization()

        # Compute adapted weights
        query_weight = self.query.weight + delta_q
        key_weight = self.key.weight + delta_k
        value_weight = self.value.weight + delta_v

        # Project using adapted weights
        mixed_query = F.linear(hidden_states, query_weight, self.query.bias)
        key = F.linear(hidden_states, key_weight, self.key.bias)
        value = F.linear(hidden_states, value_weight, self.value.bias)

        # Original attention computation
        key = self.transpose_for_scores(key)
        value = self.transpose_for_scores(value)
        query = self.transpose_for_scores(mixed_query)

        context = F.scaled_dot_product_attention(
            query, key, value,
            dropout_p=self.attention_probs_dropout_prob if self.training else 0.0,
            is_causal=False
        )

        # Rest of original forward pass
        context = context.permute(0, 2, 1, 3).contiguous()
        context = context.view(context.size()[:-2] + (self.all_head_size,))

        return (context,)

    def test_mode(self, zero_deltas: bool = True):
        """For validation/testing: disable deltas or set to zero"""
        if zero_deltas:
            self.factorization.reset_deltas_to_zero()
        self.factorization.requires_grad_(not zero_deltas)

In [ ]:
from copy import deepcopy
model = deepcopy(pretrained_model)

In [ ]:
def replace_attention_layers(model):
    for i in range(len(model.vit.encoder.layer)):
        original_layer = model.vit.encoder.layer[i]

        # Create new attention layer with factorization
        new_attention = ModifiedViTSdpaSelfAttention(model.config)

        # Copy original weights (Q/K/V) to new layer
        new_attention.query.load_state_dict(original_layer.attention.attention.query.state_dict())
        new_attention.key.load_state_dict(original_layer.attention.attention.key.state_dict())
        new_attention.value.load_state_dict(original_layer.attention.attention.value.state_dict())

        # Preserve output layer weights
        new_output = deepcopy(original_layer.attention.output)

        # Rebuild the attention module
        new_layer = deepcopy(original_layer)
        new_layer.attention.attention = new_attention
        new_layer.attention.output = new_output

        model.vit.encoder.layer[i] = new_layer
    return model

# Apply to your pretrained model
model = replace_attention_layers(model)

In [ ]:
def initialize_zero_deltas(model):
    for layer in model.vit.encoder.layer:
        if hasattr(layer.attention.attention, 'factorization'):
            layer.attention.attention.factorization.reset_deltas_to_zero()
    return model

model = initialize_zero_deltas(model)

In [ ]:
# Before/after test
with torch.no_grad():
    original_output = pretrained_model(**inputs)
    new_output = model(**inputs)

assert torch.allclose(original_output.logits, new_output.logits, atol=1e-6)
print("Outputs match! Model upgrade successful.")

Outputs match! Model upgrade successful.


In [ ]:
# 1. Disable zero initialization (keep original factorization parameters)
#    (Skip the `initialize_zero_deltas` step entirely)

# 2. Compare outputs before/after factorization
with torch.no_grad():
    original_output = pretrained_model(**inputs)
    new_output = model(**inputs)  # Uses initialized (non-zero) deltas

# 3. Verify outputs are DIFFERENT
assert not torch.allclose(
    original_output.logits,
    new_output.logits,
    atol=1e-4  # Adjust tolerance as needed
), "Outputs should differ when deltas are non-zero!"

print("Outputs differ! Factorization is active.")

Outputs differ! Factorization is active.


In [ ]:
predicted_label = new_output.logits.argmax(-1).item()
print(pretrained_model.config.id2label[predicted_label])

Egyptian cat


In [ ]:
# # Freeze all parameters in the model
# for param in model.parameters():
#     param.requires_grad = False

# # Unfreeze factorization parameters
# for layer in model.vit.encoder.layer:
#     attention = layer.attention.attention  # ViTSdpaSelfAttention instance
#     if hasattr(attention, 'factorization'):
#         for param in attention.factorization.parameters():
#             param.requires_grad = True  # Only these will be updated

In [ ]:
# Define transforms (ViT expects 224x224 images)
train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
    std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225])
])

# Download dataset
train_dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=test_transform
)

# Create dataloaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

100%|██████████| 170M/170M [00:02<00:00, 57.5MB/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified


In [ ]:
# Replace final classifier (original was for ImageNet's 1000 classes)
model.classifier = nn.Linear(
    in_features=768,  # ViT hidden size
    out_features=10   # CIFAR-10 classes
)

# Unfreeze classifier + factorization parameters
for param in model.parameters():
    param.requires_grad = False  # Freeze all first

for layer in model.vit.encoder.layer:
    if hasattr(layer.attention.attention, 'factorization'):
        for param in layer.attention.attention.factorization.parameters():
            param.requires_grad = True

model.classifier.requires_grad_(True)  # Unfreeze classifier

Linear(in_features=768, out_features=10, bias=True)

In [ ]:
def print_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable: {trainable} / Total: {total} ({100*trainable/total:.2f}%)")

# Check factorization is the only trainable component
print_trainable_params(model)

Trainable: 147744 / Total: 86715400 (0.17%)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=3e-4,
    weight_decay=1e-4
)
criterion = nn.CrossEntropyLoss()

# Training loop
NUM_EPOCHS = 5
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Epoch {epoch+1} | Batch {batch_idx} | Loss: {loss.item():.4f}")

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"\nEpoch {epoch+1} | Avg Loss: {total_loss/len(train_loader):.4f}")
    print(f"Validation Accuracy: {100*correct/total:.2f}%\n")

Epoch 1 | Batch 0 | Loss: 2.5500


KeyboardInterrupt: 